# 06. Мост Python → R: передача подготовленных данных

## Тема

**Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных**

В предыдущих ноутбуках мы:

- загрузили данные из разных форматов;
- проверили типы и качество;
- объединили несколько источников;
- провели базовый анализ;
- построили графики.

Теперь покажем, как результат Python-обработки можно передать в R.

В этом ноутбуке Python сохранит подготовленные данные в форматы:

1. CSV;
2. Excel;
3. Parquet;
4. DuckDB.

Затем посмотрим, как R может прочитать каждый из этих форматов.

## 1. Цель ноутбука

После выполнения ноутбука вы должны понимать:

1. Зачем передавать данные из Python в R.
2. Как Python сохраняет таблицу в CSV.
3. Как Python сохраняет таблицу в Excel.
4. Как Python сохраняет таблицу в Parquet.
5. Как Python сохраняет таблицу в DuckDB.
6. Как R читает CSV через `readr`.
7. Как R читает Excel через `readxl`.
8. Как R читает Parquet через `arrow`.
9. Как R подключается к DuckDB через `DBI` и `duckdb`.
10. Как выбрать формат обмена в простой учебной задаче.

Главная идея:

> Python и R не конкурируют в этом блоке. Они могут быть частями одного аналитического процесса.

## 2. Когда нужен мост Python → R

Передача данных из Python в R полезна, если:

- Python используется для загрузки, очистки и интеграции данных;
- R используется для статистического анализа;
- команда работает в разных инструментах;
- нужно передать результат в RStudio, Positron или Quarto;
- нужно сохранить подготовленную таблицу в переносимом формате.

Пример процесса:

```text
Python
загрузка → очистка → объединение → расчет показателей
↓
CSV / Excel / Parquet / DuckDB
↓
R
статистика → визуализация → отчет
```

## 3. Форматы обмена: краткое сравнение

| Формат | Когда использовать | Плюсы | Ограничения |
|---|---|---|---|
| CSV | Простая передача таблицы | Максимально простой и понятный | Теряются типы дат, нет нескольких листов |
| Excel | Передача людям и ручной просмотр | Удобно открыть в офисном ПО | Не лучший формат для больших данных |
| Parquet | Аналитические данные и большие таблицы | Хранит типы, компактный, быстрый | Нужны библиотеки `pyarrow` / `arrow` |
| DuckDB | Локальная аналитическая база | Можно читать SQL-запросами из Python и R | Нужно понимать базовую работу с БД |

На занятии мы покажем все четыре варианта, но без глубокого погружения в R.

## 4. Импорт библиотек Python

Используем:

- `pandas` — работа с таблицей;
- `Path` — пути к файлам;
- `duckdb` — локальная аналитическая база.

In [ ]:
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

## 5. Проверка дополнительных библиотек

Для Parquet нужен `pyarrow`.  
Для DuckDB нужен пакет `duckdb`.

Если появилась ошибка импорта, установите зависимости:

```bash
pip install -r requirements.txt
```

In [ ]:
try:
    import pyarrow
    print("pyarrow:", pyarrow.__version__)
except ImportError:
    print("pyarrow не установлен. Установите: pip install pyarrow")

try:
    import duckdb
    print("duckdb:", duckdb.__version__)
except ImportError:
    print("duckdb не установлен. Установите: pip install duckdb")

# Часть 1. Загрузка подготовленной таблицы

## 6. Поиск `sales_prepared.csv`

Нам нужен файл:

```text
data/prepared/sales_prepared.csv
```

Если он не найден, значит нужно сначала выполнить:

```text
03_data_integration.ipynb
```

В этом учебном ноутбуке добавлен резервный сценарий: если файл отсутствует, Python попробует собрать его из исходных данных.

In [ ]:
def find_prepared_file() -> Path:
    """Найти файл sales_prepared.csv в типовых местах."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "prepared" / "sales_prepared.csv",
        current_dir.parent / "data" / "prepared" / "sales_prepared.csv",
        current_dir.parent.parent / "data" / "prepared" / "sales_prepared.csv",
        current_dir / "sales_prepared.csv",
        current_dir.parent / "sales_prepared.csv",
        Path("/mnt/data/data/prepared/sales_prepared.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return current_dir / "data" / "prepared" / "sales_prepared.csv"


prepared_path = find_prepared_file()

print("Путь к sales_prepared.csv:")
print(prepared_path)
print("Файл существует:", prepared_path.exists())

## 7. Резервная сборка `sales_prepared.csv`

Эта ячейка нужна для устойчивости занятия.  
Если `sales_prepared.csv` уже есть, она ничего не пересобирает.

In [ ]:
def find_data_dir() -> Path:
    """Найти папку с исходными учебными данными."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "raw",
        current_dir.parent / "data" / "raw",
        current_dir.parent.parent / "data" / "raw",
    ]

    for candidate in candidates:
        if (candidate / "sales.csv").exists():
            return candidate

    return current_dir / "data" / "raw"


def parse_dates_safely(series: pd.Series) -> pd.Series:
    """Преобразовать даты с учетом разных версий pandas."""
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(series, errors="coerce", dayfirst=True)


def build_sales_prepared_if_missing(prepared_path: Path) -> Path:
    """Собрать sales_prepared.csv из исходных файлов, если он отсутствует."""
    if prepared_path.exists():
        print("sales_prepared.csv уже существует. Пересборка не требуется.")
        return prepared_path

    data_dir = find_data_dir()
    print("sales_prepared.csv не найден. Собираем из исходных файлов.")
    print("Папка исходных данных:", data_dir)

    sales = pd.read_csv(data_dir / "sales.csv")
    products = pd.read_excel(data_dir / "products.xlsx", sheet_name="products")
    regions = pd.read_json(data_dir / "regions.json")
    clients = pd.read_csv(data_dir / "clients.csv")
    plans = pd.read_html(data_dir / "web_table_sample.html")[0]

    sales["order_date"] = parse_dates_safely(sales["order_date"])
    sales["month"] = sales["order_date"].dt.to_period("M").astype(str)
    sales["channel"] = sales["channel"].astype("string").str.strip().str.lower()
    sales["quantity"] = pd.to_numeric(sales["quantity"], errors="coerce")
    sales["unit_price"] = pd.to_numeric(sales["unit_price"], errors="coerce")
    sales["discount_percent"] = pd.to_numeric(sales["discount_percent"], errors="coerce").fillna(0)
    sales = sales.drop_duplicates(subset=["sale_id"], keep="first")

    products["category"] = products["category"].astype("string").str.strip().str.lower()
    products["purchase_price"] = pd.to_numeric(products["purchase_price"], errors="coerce")
    products = products.drop_duplicates(subset=["product_id"], keep="first")

    regions["federal_district"] = regions["federal_district"].astype("string").str.strip().str.lower()

    clients["client_type"] = clients["client_type"].astype("string").str.strip().str.upper()
    clients["registration_date"] = parse_dates_safely(clients["registration_date"])
    clients = clients.drop_duplicates(subset=["client_id"], keep="first")

    plans["channel"] = plans["channel"].astype("string").str.strip().str.lower()
    plans["sales_plan"] = pd.to_numeric(plans["sales_plan"], errors="coerce")
    plans["orders_plan"] = pd.to_numeric(plans["orders_plan"], errors="coerce")
    plans["month_dt"] = parse_dates_safely(plans["month"].astype("string"))
    plans["month"] = plans["month_dt"].dt.to_period("M").astype(str)
    plans = plans.drop(columns=["month_dt"])

    prepared = (
        sales
        .merge(products, on="product_id", how="left", validate="many_to_one")
        .merge(regions, on="region_id", how="left", validate="many_to_one")
        .merge(clients, on="client_id", how="left", validate="many_to_one")
        .merge(plans, on=["region_id", "channel", "month"], how="left", validate="many_to_one")
    )

    prepared["gross_revenue"] = prepared["quantity"] * prepared["unit_price"]
    prepared["discount_amount"] = prepared["gross_revenue"] * prepared["discount_percent"] / 100
    prepared["net_revenue"] = prepared["gross_revenue"] - prepared["discount_amount"]
    prepared["purchase_cost"] = prepared["quantity"] * prepared["purchase_price"]
    prepared["gross_profit"] = prepared["net_revenue"] - prepared["purchase_cost"]
    prepared["plan_completion_rate"] = prepared["net_revenue"] / prepared["sales_plan"]

    prepared_path.parent.mkdir(parents=True, exist_ok=True)
    prepared.to_csv(prepared_path, index=False, encoding="utf-8")

    print("Файл создан:", prepared_path)
    return prepared_path


prepared_path = build_sales_prepared_if_missing(prepared_path)

## 8. Загружаем подготовленные данные

In [ ]:
sales_prepared = pd.read_csv(prepared_path)

print("Размер таблицы:", sales_prepared.shape)

sales_prepared.head()

## 9. Минимальная проверка перед экспортом

Перед передачей в R проверим:

- размер таблицы;
- ключевые поля;
- наличие выручки и прибыли.

In [ ]:
print("Размер:", sales_prepared.shape)

important_columns = [
    "sale_id",
    "order_date",
    "product_name",
    "category",
    "region_name",
    "client_type",
    "channel",
    "net_revenue",
    "gross_profit",
]

available_columns = [col for col in important_columns if col in sales_prepared.columns]

sales_prepared[available_columns].head()

In [ ]:
print("Пропуски в ключевых аналитических полях:")
sales_prepared[available_columns].isna().sum()

# Часть 2. Python сохраняет данные для R

## 10. Создаем папку для обмена

Все файлы для передачи в R сохраним в отдельную папку:

```text
data/output/python_to_r
```

In [ ]:
exchange_dir = Path("data/output/python_to_r")
exchange_dir.mkdir(parents=True, exist_ok=True)

print("Папка обмена:")
print(exchange_dir)

## 11. Выберем компактный набор столбцов

Для учебного R-блока не нужно передавать все столбцы.  
Оставим основные аналитические поля.

In [ ]:
bridge_columns = [
    "sale_id",
    "order_date",
    "month",
    "product_id",
    "product_name",
    "category",
    "region_id",
    "region_name",
    "federal_district",
    "client_id",
    "client_type",
    "channel",
    "quantity",
    "unit_price",
    "discount_percent",
    "net_revenue",
    "gross_profit",
    "sales_plan",
    "plan_completion_rate",
]

bridge_columns = [col for col in bridge_columns if col in sales_prepared.columns]

bridge_df = sales_prepared[bridge_columns].copy()

print("Размер bridge_df:", bridge_df.shape)

bridge_df.head()

## 12. Сохранение в CSV

CSV — самый простой формат обмена.

Плюсы:

- легко открыть почти везде;
- легко прочитать в Python и R;
- удобно для небольших таблиц.

Ограничения:

- типы данных сохраняются не идеально;
- даты часто приходится преобразовывать заново;
- нет нескольких листов.

In [ ]:
csv_path = exchange_dir / "sales_prepared_for_r.csv"

bridge_df.to_csv(csv_path, index=False, encoding="utf-8")

print("CSV сохранен:")
print(csv_path)

print("Файл существует:", csv_path.exists())

## 13. Сохранение в Excel

Excel удобен, если результат нужно открыть вручную или передать коллегам.

Плюсы:

- привычный формат;
- можно хранить несколько листов;
- удобно проверять глазами.

Ограничения:

- не лучший вариант для больших данных;
- может быть медленнее CSV/Parquet;
- не всегда стабильно сохраняет типы.

In [ ]:
excel_path = exchange_dir / "sales_prepared_for_r.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    bridge_df.to_excel(writer, sheet_name="sales_prepared", index=False)

    summary_by_category = (
        bridge_df
        .groupby("category", dropna=False)
        .agg(
            rows_count=("sale_id", "count"),
            total_revenue=("net_revenue", "sum"),
            total_profit=("gross_profit", "sum"),
        )
        .reset_index()
        .sort_values("total_revenue", ascending=False)
    )

    summary_by_category.to_excel(writer, sheet_name="summary_category", index=False)

print("Excel сохранен:")
print(excel_path)

print("Файл существует:", excel_path.exists())

## 14. Сохранение в Parquet

Parquet — аналитический колоночный формат.

Плюсы:

- хорошо подходит для аналитических данных;
- обычно компактнее CSV;
- лучше сохраняет типы;
- быстро читается инструментами анализа.

Ограничение:

- нужны библиотеки `pyarrow` в Python и `arrow` в R.

In [ ]:
parquet_path = exchange_dir / "sales_prepared_for_r.parquet"

bridge_df.to_parquet(parquet_path, index=False, engine="pyarrow")

print("Parquet сохранен:")
print(parquet_path)

print("Файл существует:", parquet_path.exists())

## 15. Сохранение в DuckDB

DuckDB — это локальная аналитическая база данных в одном файле.

Плюсы:

- данные можно читать SQL-запросами;
- один и тот же `.duckdb`-файл можно открыть из Python и R;
- удобно для аналитических витрин;
- можно хранить несколько таблиц.

Создадим файл:

```text
analytics.duckdb
```

и таблицу:

```text
sales_prepared
```

In [ ]:
import duckdb

duckdb_path = exchange_dir / "analytics.duckdb"

connection = duckdb.connect(str(duckdb_path))

connection.execute("DROP TABLE IF EXISTS sales_prepared")
connection.register("bridge_df_view", bridge_df)
connection.execute("CREATE TABLE sales_prepared AS SELECT * FROM bridge_df_view")
connection.unregister("bridge_df_view")

# Добавим агрегированную таблицу как отдельную таблицу базы.
connection.execute("DROP TABLE IF EXISTS summary_by_category")
connection.register("summary_by_category_view", summary_by_category)
connection.execute("CREATE TABLE summary_by_category AS SELECT * FROM summary_by_category_view")
connection.unregister("summary_by_category_view")

print("Таблицы в DuckDB:")
display(connection.execute("SHOW TABLES").df())

connection.close()

print("DuckDB сохранен:")
print(duckdb_path)

print("Файл существует:", duckdb_path.exists())

# Часть 3. Проверка файлов, созданных Python

## 16. Список файлов обмена

Проверим, что Python создал все четыре формата.

In [ ]:
expected_outputs = [
    csv_path,
    excel_path,
    parquet_path,
    duckdb_path,
]

for path in expected_outputs:
    print(f"{path.name:<32} exists={path.exists()}")

## 17. Проверка чтения обратно в Python

Это не обязательный этап, но полезно проверить, что файлы действительно читаются.

In [ ]:
csv_check = pd.read_csv(csv_path)
excel_check = pd.read_excel(excel_path, sheet_name="sales_prepared")
parquet_check = pd.read_parquet(parquet_path, engine="pyarrow")

print("CSV:", csv_check.shape)
print("Excel:", excel_check.shape)
print("Parquet:", parquet_check.shape)

In [ ]:
connection = duckdb.connect(str(duckdb_path), read_only=True)

duckdb_check = connection.execute("""
SELECT 
    COUNT(*) AS rows_count,
    SUM(net_revenue) AS total_revenue,
    SUM(gross_profit) AS total_profit
FROM sales_prepared
""").df()

connection.close()

duckdb_check

# Часть 4. R читает файлы, созданные Python

## 18. Где запускать R-код

R-код ниже можно запускать:

- в RStudio;
- в Positron;
- в R-консоли;
- в Quarto-документе;
- в Jupyter Notebook с R-kernel.

В этом Python-ноутбуке R-код дан как готовые блоки для копирования.

Важно: пути к файлам должны совпадать с вашей структурой проекта.

## 19. Установка R-пакетов

Один раз установите нужные R-пакеты:

```r
install.packages(c(
  "readr",
  "readxl",
  "arrow",
  "DBI",
  "duckdb",
  "dplyr",
  "ggplot2"
))
```

Назначение пакетов:

| Пакет | Зачем нужен |
|---|---|
| `readr` | читать CSV |
| `readxl` | читать Excel |
| `arrow` | читать Parquet |
| `DBI` | общий интерфейс к базам данных |
| `duckdb` | подключаться к DuckDB-файлу |
| `dplyr` | группировки и обработка таблиц |
| `ggplot2` | графики в R |

## 20. R читает CSV

```r
library(readr)

csv_path <- "data/output/python_to_r/sales_prepared_for_r.csv"

sales_csv <- read_csv(csv_path)

glimpse(sales_csv)
head(sales_csv)
```

Что должно получиться:

- R прочитает CSV в таблицу;
- можно посмотреть структуру через `glimpse`;
- даты могут потребовать дополнительного преобразования.

## 21. R читает Excel

```r
library(readxl)

excel_path <- "data/output/python_to_r/sales_prepared_for_r.xlsx"

sales_excel <- read_excel(excel_path, sheet = "sales_prepared")

head(sales_excel)
```

Можно также прочитать лист с агрегатами:

```r
summary_category <- read_excel(excel_path, sheet = "summary_category")

summary_category
```

## 22. R читает Parquet

```r
library(arrow)

parquet_path <- "data/output/python_to_r/sales_prepared_for_r.parquet"

sales_parquet <- read_parquet(parquet_path)

head(sales_parquet)
```

Parquet удобен для аналитических данных, потому что лучше сохраняет типы и обычно эффективнее CSV.

## 23. R читает DuckDB

```r
library(DBI)
library(duckdb)

duckdb_path <- "data/output/python_to_r/analytics.duckdb"

con <- dbConnect(duckdb::duckdb(), dbdir = duckdb_path, read_only = TRUE)

dbListTables(con)

sales_from_db <- dbGetQuery(con, "
  SELECT 
    category,
    COUNT(*) AS rows_count,
    SUM(net_revenue) AS total_revenue,
    SUM(gross_profit) AS total_profit
  FROM sales_prepared
  GROUP BY category
  ORDER BY total_revenue DESC
")

sales_from_db

dbDisconnect(con, shutdown = TRUE)
```

Что происходит:

- R подключается к файлу `analytics.duckdb`;
- смотрит список таблиц;
- выполняет SQL-запрос;
- получает результат как R data frame.

## 24. R делает простой анализ после чтения

Пример после чтения CSV:

```r
library(readr)
library(dplyr)

sales_csv <- read_csv("data/output/python_to_r/sales_prepared_for_r.csv")

category_summary <- sales_csv |>
  group_by(category) |>
  summarise(
    orders_count = n(),
    total_revenue = sum(net_revenue, na.rm = TRUE),
    total_profit = sum(gross_profit, na.rm = TRUE),
    .groups = "drop"
  ) |>
  arrange(desc(total_revenue))

category_summary
```

## 25. R строит простой график

```r
library(readr)
library(dplyr)
library(ggplot2)

sales_csv <- read_csv("data/output/python_to_r/sales_prepared_for_r.csv")

category_summary <- sales_csv |>
  group_by(category) |>
  summarise(
    total_revenue = sum(net_revenue, na.rm = TRUE),
    .groups = "drop"
  )

ggplot(category_summary, aes(x = reorder(category, total_revenue), y = total_revenue)) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Выручка по категориям",
    x = "Категория",
    y = "Выручка"
  )
```

Это не основной блок занятия, а демонстрация: подготовленные Python-данные можно продолжить анализировать в R.

# Часть 5. Как выбрать формат для передачи

## 26. Практические рекомендации

| Ситуация | Рекомендуемый формат |
|---|---|
| Нужно быстро передать маленькую таблицу | CSV |
| Нужно открыть файл вручную и показать коллегам | Excel |
| Нужно передать аналитические данные с сохранением типов | Parquet |
| Нужно работать SQL-запросами из Python и R | DuckDB |
| Нужно сделать отчет для людей | Excel + графики |
| Нужно сделать воспроизводимый аналитический пайплайн | Parquet или DuckDB |

Для начинающих достаточно понять:

> CSV — самый простой, Excel — самый привычный, Parquet — аналитический, DuckDB — база данных в одном файле.

## 27. Частые ошибки при передаче Python → R

### Ошибка 1. Неверный путь к файлу

В R проверьте рабочую папку:

```r
getwd()
list.files()
```

### Ошибка 2. R-пакет не установлен

Установите пакет:

```r
install.packages("readr")
```

### Ошибка 3. CSV прочитался, но даты стали текстом

Преобразуйте дату в R:

```r
sales_csv$order_date <- as.Date(sales_csv$order_date)
```

### Ошибка 4. Parquet не читается

Проверьте, установлен ли пакет `arrow`:

```r
install.packages("arrow")
library(arrow)
```

### Ошибка 5. DuckDB-файл не найден

Проверьте путь:

```r
file.exists("data/output/python_to_r/analytics.duckdb")
```

### Ошибка 6. DuckDB занят другим процессом

Закройте подключение в Python или R:

```r
dbDisconnect(con, shutdown = TRUE)
```

# Часть 6. Мини-задания

## Задание 1

В Python сохраните `bridge_df` в CSV с именем:

```text
data/output/python_to_r/task_sales.csv
```

In [ ]:
# Ваш код здесь

## Задание 2

В Python сохраните `summary_by_category` в Excel с именем:

```text
data/output/python_to_r/task_summary.xlsx
```

In [ ]:
# Ваш код здесь

## Задание 3

В Python сохраните `bridge_df` в Parquet с именем:

```text
data/output/python_to_r/task_sales.parquet
```

In [ ]:
# Ваш код здесь

## Задание 4

В Python создайте DuckDB-файл:

```text
data/output/python_to_r/task_analytics.duckdb
```

и сохраните в него таблицу `bridge_df` под именем:

```text
sales_task
```

In [ ]:
# Ваш код здесь

## Задание 5

Напишите R-код, который читает `task_sales.csv`.

Подсказка:

```r
library(readr)

task_sales <- read_csv("data/output/python_to_r/task_sales.csv")

head(task_sales)
```

## Задание 6

Напишите R-код, который подключается к `task_analytics.duckdb` и считает выручку по категориям.

Подсказка:

```r
library(DBI)
library(duckdb)

con <- dbConnect(duckdb::duckdb(), dbdir = "data/output/python_to_r/task_analytics.duckdb")

result <- dbGetQuery(con, "
  SELECT category, SUM(net_revenue) AS total_revenue
  FROM sales_task
  GROUP BY category
  ORDER BY total_revenue DESC
")

result

dbDisconnect(con, shutdown = TRUE)
```

# Часть 7. Контрольные вопросы

Ответьте своими словами:

1. Зачем передавать данные из Python в R?
2. Чем CSV удобен для обмена?
3. Почему CSV может быть неудобен для дат и типов данных?
4. Когда лучше использовать Excel?
5. Для чего нужен Parquet?
6. Чем DuckDB отличается от CSV-файла?
7. Почему DuckDB удобно использовать между Python и R?
8. Какой R-пакет читает CSV?
9. Какой R-пакет читает Excel?
10. Какой R-пакет читает Parquet?
11. Какие R-пакеты нужны для подключения к DuckDB?
12. Какой формат вы бы выбрали для маленькой таблицы?
13. Какой формат вы бы выбрали для аналитической витрины?

# 28. Итог ноутбука

В этом ноутбуке мы показали вводный мост Python → R.

Python сохранил подготовленные данные в форматы:

```text
CSV
Excel
Parquet
DuckDB
```

R может прочитать эти данные через:

```text
readr::read_csv()
readxl::read_excel()
arrow::read_parquet()
DBI + duckdb
```

Главный вывод:

> Python можно использовать для подготовки и интеграции данных, а R — для дальнейшего статистического анализа, визуализации или отчетности. Важно не противопоставлять инструменты, а понимать, как передавать данные между ними.